In [1]:
pip install transformers torch pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.optim import AdamW  # <--- Use this instead of transformers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [3]:
# Load Dataset
# ----------------------------
columns = ["target", "ids", "date", "flag", "user", "text"]
df = pd.read_csv("training.1600000.processed.noemoticon.csv", encoding="ISO-8859-1", names=columns)

In [4]:
# 2. Ab "target" column ko encode karein
encoder = LabelEncoder()
df["label"] = encoder.fit_transform(df["target"])

# 3. Check karein ke sab sahi chal gaya
print("Columns in DataFrame:", df.columns.tolist())
print("\nFirst 5 rows:")
print(df[["text", "label"]].head())

Columns in DataFrame: ['target', 'ids', 'date', 'flag', 'user', 'text', 'label']

First 5 rows:
                                                text  label
0  @switchfoot http://twitpic.com/2y1zl - Awww, t...      0
1  is upset that he can't update his Facebook by ...      0
2  @Kenichan I dived many times for the ball. Man...      0
3    my whole body feels itchy and like its on fire       0
4  @nationwideclass no, it's not behaving at all....      0


In [5]:
# 20,000 ki jagah sirf 2,000 samples lein (Total 4,000)
df_pos = df[df['label'] == 1].sample(2000, random_state=42)
df_neg = df[df['label'] == 0].sample(2000, random_state=42)
df_sampled = pd.concat([df_pos, df_neg]).sample(frac=1, random_state=42).reset_index(drop=True)

In [6]:
# 2. Split Data
# ----------------------------
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df_sampled["text"],
    df_sampled["label"],
    test_size=0.2,
    random_state=42
)

In [10]:
# 3. Load BERT Tokenizer
# ----------------------------
print("Loading DistilBERT Model...")
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

Loading DistilBERT Model...


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
# ----------------------------
# 4. Create Dataset Class
# ----------------------------
class SentimentDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
 
    def __len__(self):
        return len(self.texts)
 
    def __getitem__(self, idx):
        encoding = tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=128,
            return_tensors="pt"
        )
 
        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }
 
# Create datasets
train_dataset = SentimentDataset(train_texts, train_labels)
test_dataset = SentimentDataset(test_texts, test_labels)
 
# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8)

In [13]:
# ----------------------------
# 5. Load DistilBERT Model
# ----------------------------
print("Loading DistilBERT Model...")

# BertForSequenceClassification ki jagah DistilBertForSequenceClassification likhein:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2 
)
 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Using device: {device}")

Loading DistilBERT Model...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Using device: cpu


In [14]:
# 6. Optimizer
# ----------------------------
optimizer = AdamW(model.parameters(), lr=2e-5)

In [16]:

from transformers import DistilBertTokenizer

# Tokenizer ko dobara define kar rahe hain taake NameError khatam ho jaye
print("Re-initializing Tokenizer...")
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

Re-initializing Tokenizer...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [17]:
# 7. Training Loop
# ----------------------------
epochs = 3
print("Starting Training...")
for epoch in range(epochs):
    model.train()
    total_loss = 0
 
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
 
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
 
        loss = outputs.loss
        total_loss += loss.item()
 
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
 
    print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}")
 
print("Training Complete!")

Starting Training...
Epoch 1/3 - Loss: 0.5271
Epoch 2/3 - Loss: 0.3364
Epoch 3/3 - Loss: 0.1596
Training Complete!


In [18]:
# 8. Testing the Model
# ----------------------------
print("Evaluating Model...")
model.eval()
correct = 0
total = 0
 
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
 
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
 
        predictions = torch.argmax(outputs.logits, dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
 
accuracy = correct / total
print(f"Final Accuracy: {accuracy * 100:.2f}%")

Evaluating Model...
Final Accuracy: 78.00%


In [19]:
# 9. Predict Sentiment for New Text
# ----------------------------
label_names = encoder.classes_  # Yeh automatic [0, 4] utha lega
 
text = "The product quality is excellent"
print(f"\nPredicting for text: '{text}'")

inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
inputs = {k: v.to(device) for k, v in inputs.items()}
 
with torch.no_grad():
    outputs = model(**inputs)
 
prediction = torch.argmax(outputs.logits, dim=1).item()

# Sentiment140 mein 0 = Negative aur 4 = Positive hota hai
sentiment_result = "Positive" if label_names[prediction] == 4 else "Negative"
print("Sentiment Result:", sentiment_result)


Predicting for text: 'The product quality is excellent'
Sentiment Result: Positive
